In [ ]:
import pandas as pd
import matplotlib.ticker as ticker 
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from datetime import date
from xbbg import blp # Bloomberg Python API

## Introduction
In this notebook we build an on-demand trade cost analysis for US equity orders. We will use component fill data and the Bloomberg Desktop API to look at:
*   Completion Rate
*   Percentage of Volume
*   Implementation Shortfall
*   Interval VWAP
*   Venue Analysis

All data is random and not representative of a real-world trade. This is not investment advice.

In [ ]:
# Read in the component fill, symbol, and side data
fill_data = pd.read_excel('fills.xlsx', sheet_name='Data')
bbg_sym = fill_data.loc[0,'SYMBOL'] + " US Equity"
side = fill_data.loc[0,'SIDE']

## Completion Rate
*   Percentage of order quantity filled, measured in 1-minute buckets.

In [ ]:
pct_complete = fill_data[['Cli Ord Cum Qty', 'Client Ord Qty', 'Exec Date Time']].copy()
pct_complete['Exec Date Time'] = pd.to_datetime(pct_complete['Exec Date Time'])
pct_complete['Pct Complete'] = (pct_complete["Cli Ord Cum Qty"] / pct_complete["Client Ord Qty"]) * 100
pct_complete = pct_complete.sort_values(by=['Pct Complete'], ascending=True)

pct_complete["Minutes From Arrival"] = ""
for i in pct_complete.index:
    pct_complete.loc[i, "Minutes From Arrival"] = (pct_complete.loc[i, 'Exec Date Time'] - pct_complete['Exec Date Time'].min()).total_seconds() / 60.0

font_title = {'color':'black', 'size':14}
font = {'family' : 'Times New Roman',
        'weight' : 'normal',
        'size'   : 12}

x = pct_complete['Minutes From Arrival']
y = pct_complete['Pct Complete']

first_print = pct_complete['Exec Date Time'].min().strftime("%H:%M:%S")
last_print  = pct_complete['Exec Date Time'].max().strftime("%H:%M:%S")

fig, ax = plt.subplots(nrows= 1, ncols= 1)

ax.plot(x, y, color= '#04caf4')
ax.set(xlabel= 'Minutes from Arrival'+ '\n\nFirst Print: '+ first_print+ '\nLast Print: '+ last_print)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(decimals= 0))
ax.set_ylim(0, pct_complete['Pct Complete'].max())
ax.set_xlim(0, pct_complete['Minutes From Arrival'].max())

plt.title('Completion Rate | '+ bbg_sym, fontdict= font_title)
plt.yticks([0,(pct_complete['Pct Complete'].max()/2),pct_complete['Pct Complete'].max()])
plt.grid(linewidth= 0.25)
plt.rc('font', **font)

## Percentage of Volume
*    Shares executed as a percentage of total market volume, measured in 1-minute buckets.

In [ ]:
bbg_data = blp.bdib(ticker=bbg_sym, dt=date.today()).reset_index(col_level=1).rename(columns={'index':'Exec Date Time'})
bbg_data.columns = bbg_data.columns.droplevel()

# Get the time series data from the original Dataframe, localize the Time Stamps to match the Bloomberg formatting
order_vol_by_minute = fill_data[['Exec Date Time','NUM_SHARES']].copy()
order_vol_by_minute['Exec Date Time'] = pd.to_datetime(order_vol_by_minute['Exec Date Time'])
order_vol_by_minute['Exec Date Time'] = order_vol_by_minute['Exec Date Time'].dt.tz_localize('US/Eastern')
order_vol_by_minute['Exec Date Time'] = order_vol_by_minute['Exec Date Time'].dt.tz_convert('America/New_York')

first_print = order_vol_by_minute['Exec Date Time'].min().strftime("%H:%M:%S")
last_print = order_vol_by_minute['Exec Date Time'].max().strftime("%H:%M:%S")

# Bucket the order fill time series data into 1-minute intervals
order_vol_by_minute = order_vol_by_minute.groupby(["Exec Date Time"]).NUM_SHARES.sum().reset_index()
order_vol_by_minute = order_vol_by_minute.set_index(["Exec Date Time"])
order_vol_by_minute = order_vol_by_minute.resample("1Min").sum().reset_index()

# Synchornize the Bloomberg and order time series to the same start time
bbg_data = bbg_data[bbg_data['Exec Date Time'] >= order_vol_by_minute['Exec Date Time'].min()] 

bbg_data['Volume Cumulative Sum'] = bbg_data['volume'].cumsum()
merged_df = pd.merge(bbg_data, order_vol_by_minute, on='Exec Date Time')
merged_df['Order Volume Cumulative Sum'] = merged_df['NUM_SHARES'].cumsum()
merged_df['POV'] = (merged_df['Order Volume Cumulative Sum'] / merged_df['Volume Cumulative Sum']) *100

merged_df["Minutes From Arrival"] = ""
for i in merged_df.index:
    merged_df.loc[i, "Minutes From Arrival"] = (merged_df.loc[i, 'Exec Date Time'] - merged_df['Exec Date Time'].min()).total_seconds() / 60.0

x = merged_df['Minutes From Arrival']
y = merged_df['POV']

fig, ax = plt.subplots(nrows=1,ncols=1)

ax.plot(x,y,color='#3b00fd')
ax.set(xlabel='Minutes from Arrival'+ 
       '\n\nRange: ' + str(round(merged_df['POV'].min(), 2)) + " - " + str(round(merged_df['POV'].max(), 2)) + ' percent')
ax.set_ylim(merged_df['POV'].min(), merged_df['POV'].max())
ax.set_xlim(0, merged_df['Minutes From Arrival'].max())
ax.yaxis.set_major_formatter(ticker.PercentFormatter(decimals=1))

plt.title('Percentage of Volume | '+ bbg_sym, fontdict= font_title)
plt.grid(linewidth= 0.25)
plt.rc('font', **font)

## Implementation Shortfall
*   Simple return of rolling weighted average execution price vs arrival price

In [ ]:
avg_px = fill_data[['Exec Date Time', 'Cli Ord Cum Qty', 'Cl Ord Avg Price']].copy().sort_values(by=['Exec Date Time'], ascending=[True]).reset_index()
avg_px['Exec Date Time'] = pd.to_datetime(avg_px['Exec Date Time'])
avg_px['Exec Date Time'] = avg_px['Exec Date Time'].dt.tz_localize('US/Eastern')
avg_px['Exec Date Time'] = avg_px['Exec Date Time'].dt.tz_convert('America/New_York')

arrival_px = avg_px.loc[0, 'Cl Ord Avg Price']

avg_px = avg_px.set_index(["Exec Date Time"])
avg_px = avg_px.resample("1Min").last().reset_index().ffill()
avg_px["Minutes From Arrival"] = ""

for i in avg_px.index:
    avg_px.loc[i, "Minutes From Arrival"] = (avg_px.loc[i, 'Exec Date Time'] - avg_px['Exec Date Time'].min()).total_seconds() / 60.0

if side == 'BUY':
    avg_px['return'] = (arrival_px - avg_px['Cl Ord Avg Price']) / arrival_px
elif side == 'SELL':
    avg_px['return'] = (avg_px['Cl Ord Avg Price'] - arrival_px) / arrival_px

fig, ax = plt.subplots()
x = avg_px['Minutes From Arrival']
y = avg_px['return']

ax.plot(x, y, color='#ff007c', lw=1.1)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:.4f}'))
ax.set(xlabel='Minutes from Arrival', ylabel='Simple Return from Arrival (%)')
ax.set_xlim(0, merged_df['Minutes From Arrival'].max())

plt.title('Implementation Shortfall | '+ bbg_sym, fontdict= font_title)
plt.grid(linewidth= 0.25)
plt.rc('font', **font)

## Interval VWAP
*  Rolling weighted average execution price vs market IVWAP, measured in 1-minute buckets.

In [ ]:
cum_qty = fill_data[['Exec Date Time','Cli Ord Cum Qty']].copy()
cum_qty['Exec Date Time'] = pd.to_datetime(cum_qty['Exec Date Time'])
cum_qty['Exec Date Time'] = cum_qty['Exec Date Time'].dt.tz_localize('US/Eastern')
cum_qty['Exec Date Time'] = cum_qty['Exec Date Time'].dt.tz_convert('America/New_York')

cum_qty = cum_qty.set_index(["Exec Date Time"])
cum_qty = cum_qty.resample("1Min").max().reset_index()
cum_qty = cum_qty.ffill()
avg_px = fill_data[['Cli Ord Cum Qty', 'Cl Ord Avg Price']].copy()

avg_px_cum_qty = pd.merge(cum_qty, avg_px, on='Cli Ord Cum Qty')

first_print = cum_qty['Exec Date Time'].min().strftime("%H:%M:%S")
last_print = cum_qty['Exec Date Time'].max().strftime("%H:%M:%S")
start_time = avg_px_cum_qty['Exec Date Time'].min()
end_time = avg_px_cum_qty['Exec Date Time'].max()

bbg_vwap = blp.bdib(ticker=bbg_sym, dt=date.today()).reset_index(col_level=1).rename(columns={'index':'Exec Date Time'})
bbg_vwap.columns = bbg_vwap.columns.droplevel()
bbg_vwap = bbg_vwap[(bbg_vwap['Exec Date Time'] >= first_print) & (bbg_vwap['Exec Date Time'] <= last_print)] 
bbg_vwap['vwap'] = bbg_vwap['value'].cumsum() / bbg_vwap['volume'].cumsum()

merged_df = pd.merge(bbg_vwap, avg_px_cum_qty, on='Exec Date Time')
merged_df["Minutes From Arrival"] = ""
for i in merged_df.index:
    merged_df.loc[i, "Minutes From Arrival"] = (merged_df.loc[i, 'Exec Date Time'] - merged_df['Exec Date Time'].min()).total_seconds() / 60.0

x = merged_df['Minutes From Arrival']
y = merged_df['vwap']
y1 = merged_df['Cl Ord Avg Price']
fig, ax = plt.subplots(nrows=1,ncols=1)

ax.plot(x,y1, label ='Avg Price',color='#0bff01',lw=1.1)
ax.plot(x,y, label = "Interval VWAP",color='#ff007c',dashes=[3,2],lw=1.1)

ax.set(xlabel='Minutes from Arrival\n\n$' + str(round(merged_df['Cl Ord Avg Price'].iloc[-1],4)) + ' -- Avg Price\n$' + str(round(merged_df['vwap'].iloc[-1],4))+ ' -- Interval VWAP')
ax.set_xlim(0, merged_df['Minutes From Arrival'].max())
ax.yaxis.set_major_formatter('${x:1.2f}')

plt.legend()
plt.title('Interval VWAP | '+ bbg_sym, fontdict= font_title)
plt.grid(linewidth= 0.25)
plt.rc('font', **font)

## Venue Analysis
*   Execution quantity broken down by venue and venue type.

In [ ]:
# Market Identifier Code, ref: https://www.iso20022.org/market-identifier-codes
mic = pd.read_csv('ISO10383_MIC.csv',index_col=0)
mic_select = mic[['MARKET NAME-SHORT NAME','TYPE']].copy().to_dict()

venue = fill_data[['Exec Qty','Exec Market']].copy()
venue['Type'] = venue['Exec Market'].map(mic_select['TYPE']).astype(str)
venue['Exec Market'] = venue['Exec Market'].map(mic_select['MARKET NAME-SHORT NAME'])

sym_qty = venue.groupby(['Exec Market']).agg({'Exec Qty': 'sum', 'Type': pd.Series.mode}).reset_index()
sym_qty = sym_qty.sort_values(by='Exec Qty',ascending=True)

font_title = {'color':'black', 'size':14}
font = {'family' : 'Times New Roman',
        'weight' : 'normal',
        'size'   : 12}

fig, ax = plt.subplots()
x = sym_qty['Exec Market']
y = sym_qty['Exec Qty']

type_colors = {'Exchange Lit':'#3b00fd',
                'Exchange Non-displayed' :'#0bff01',
                'Wholesale MM':'#ffaa00',
                'SDP':'#04caf4',
                'ATS':'#ff007c',
                'Agency Cross':'#fdfe02'}


colors = sym_qty['Type'].map(type_colors)
ax.barh(x, y, color= colors)
ax.set(xlabel='Shares Executed')

legend_elements = [Patch(facecolor=color, label=venue_type) for venue_type, color in type_colors.items()]
plt.legend(handles=legend_elements,title='Venue Type')
plt.yticks(fontsize=7)
plt.grid(axis='x',linewidth= 0.25)
plt.title('Venue Analysis | '+ bbg_sym, fontdict= font_title)
plt.show()